# ASL Word — Live Webcam Testing

This notebook lets you **test your trained ASL word model in real-time** using your webcam.

### How it works:

1. **Continuous capture** — MediaPipe extracts hand landmarks every frame (supports 1 or 2 hands)
2. **Sliding window** — buffers the last 30 frames into a sequence
3. **Prediction** — feeds the sequence to the BiLSTM model every 0.5s
4. **Sentence building** — confirmed words are appended to a sentence

### Two-Hand Support:

- **Auto-detects** the model's expected input shape (63 or 126 features)
- If the model expects **63 features** (1 hand) — uses the dominant hand only
- If the model expects **126 features** (2 hands) — captures both hands and concatenates landmarks
- Many ASL word signs require two hands for proper recognition

### Controls:

| Key         | Action                  |
| ----------- | ----------------------- |
| `q`         | Quit                    |
| `r`         | Reset sentence          |
| `SPACE`     | Add space between words |
| `BACKSPACE` | Delete last word        |

### Requirements:

- Trained model: `asl_word_lstm_model_best.h5`
- Class mapping: `asl_word_classes.csv`
- Webcam connected


In [1]:
# ===============================
# CELL 1: IMPORTS & SETUP
# ===============================

import cv2
import json
import time
import numpy as np
import pandas as pd
import mediapipe as mp
import tensorflow as tf
from pathlib import Path
from collections import deque

print(f'TensorFlow: {tf.__version__}')
print(f'OpenCV: {cv2.__version__}')
print(f'MediaPipe: {mp.__version__}')

# Check GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU detected: {gpus[0].name}')
else:
    print('⚠️ No GPU — running on CPU')


TensorFlow: 2.10.0
OpenCV: 4.11.0
MediaPipe: 0.10.9
✅ GPU detected: /physical_device:GPU:0


In [2]:
# ===============================
# CELL 2: CONFIGURATION
# ===============================

PROJECT_ROOT = Path(r'M:/Term 10/Grad')
SLR_MAIN = PROJECT_ROOT / 'SLR Main'
WORDS_ROOT = SLR_MAIN / 'Words'
OUTPUT_DIR = WORDS_ROOT / 'ASL Word (English)'
SHARED_CSV = WORDS_ROOT / 'Shared/shared_word_vocabulary.csv'

# Model files
MODEL_PATH = OUTPUT_DIR / 'asl_word_lstm_model_final.h5'
CLASSES_CSV = OUTPUT_DIR / 'asl_word_classes.csv'

# Sequence parameters (must match training)
SEQUENCE_LENGTH = 30    # frames per sequence

# Hand detection mode: auto-detected from model input shape
# - 63 features = 1 hand (21 landmarks x 3)
# - 126 features = 2 hands (2 x 21 landmarks x 3)
# Set to None for auto-detection, or override manually:
NUM_FEATURES = None  # will be set after model loads

# Live inference settings
CONFIDENCE_THRESHOLD = 0.35     # minimum confidence to accept a prediction
PREDICTION_INTERVAL = 0.5       # seconds between predictions
STABILITY_WINDOW = 3            # consecutive same predictions needed to confirm
COOLDOWN_TIME = 2.0             # seconds after confirming a word before next

# Camera
CAMERA_INDEX = 0
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720

print(f'📂 Model  : {MODEL_PATH}')
print(f'📂 Classes: {CLASSES_CSV}')
print(f'🎬 Sequence: {SEQUENCE_LENGTH} frames')
print(f'🎯 Confidence threshold: {CONFIDENCE_THRESHOLD}')
print(f'🔁 Stability window: {STABILITY_WINDOW} predictions')


📂 Model  : M:\Term 10\Grad\SLR Main\Words\ASL Word (English)\asl_word_lstm_model_final.h5
📂 Classes: M:\Term 10\Grad\SLR Main\Words\ASL Word (English)\asl_word_classes.csv
🎬 Sequence: 30 frames
🎯 Confidence threshold: 0.35
🔁 Stability window: 3 predictions


In [3]:
# ===============================
# CELL 3: LOAD MODEL & VOCABULARY
# ===============================

# --- Define the Custom Attention Layer before loading the model ---
import tensorflow as tf

class TemporalAttention(tf.keras.layers.Layer):
    """Learnable attention over time steps."""
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='att_weight', shape=(input_shape[-1], 1),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(name='att_bias', shape=(input_shape[1], 1),
                                 initializer='zeros', trainable=True)

    def call(self, x):
        e = tf.nn.tanh(tf.matmul(x, self.W) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = tf.reduce_sum(x * a, axis=1)
        return output

# --- Now load the model! ---
print('Loading model...')
# ... (Keep your existing load_model code here)

# Load model
print('Loading model...')
model = tf.keras.models.load_model(
    str(MODEL_PATH),
    custom_objects={'TemporalAttention': TemporalAttention}
)
print(f'✅ Model loaded: {model.name} — {model.count_params():,} parameters')

# Auto-detect feature count from model input shape
model_input_shape = model.input_shape  # (None, SEQUENCE_LENGTH, NUM_FEATURES)
NUM_FEATURES = model_input_shape[-1]
NUM_HANDS = 2 if NUM_FEATURES == 126 else 1
LANDMARKS_PER_HAND = 21 * 3  # 63

print(f'🖐️ Model expects {NUM_FEATURES} features → {NUM_HANDS} hand(s) mode')

# Load class mapping
class_df = pd.read_csv(CLASSES_CSV)
vocab_df = pd.read_csv(SHARED_CSV)
vocab_df = vocab_df.dropna(subset=['wlasl_class'])

id_to_english = dict(zip(vocab_df['word_id'].astype(int), vocab_df['english']))
id_to_category = dict(zip(vocab_df['word_id'].astype(int), vocab_df['category']))

# Build model_index -> word name mapping
index_to_word = {}
for _, row in class_df.iterrows():
    idx = int(row['model_class_index'])
    wid = int(row['word_id'])
    index_to_word[idx] = id_to_english.get(wid, f'word_{wid}')

num_classes = len(index_to_word)
print(f'🏷️ {num_classes} word classes loaded')
print(f'\n📋 Sample words: {list(index_to_word.values())[:15]}')


Loading model...
Loading model...


ValueError: Unrecognized keyword arguments: ['batch_shape']

In [ ]:
# ===============================
# CELL 4: MEDIAPIPE HAND DETECTOR
# ===============================
# Supports both 1-hand and 2-hand detection based on model requirements

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=NUM_HANDS,       # dynamically set based on model
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

def extract_landmarks(frame):
    """Extract hand landmarks from a single frame.

    - 1-hand mode (63 features): returns landmarks for the first detected hand.
    - 2-hand mode (126 features): returns concatenated landmarks for both hands.
      If only one hand is detected, the other hand's landmarks are zero-padded.
      Hands are ordered: Left hand first, Right hand second (consistent ordering).

    Returns: (feature_vector, list_of_hand_landmarks_for_drawing)
    """
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(rgb)

    draw_landmarks = []

    if NUM_HANDS == 1:
        # Single-hand mode (63 features)
        if results.multi_hand_landmarks:
            lm = results.multi_hand_landmarks[0]
            vec = np.array([[p.x, p.y, p.z] for p in lm.landmark], dtype=np.float32).flatten()
            draw_landmarks = [lm]
            return vec, draw_landmarks
        return np.zeros(NUM_FEATURES, dtype=np.float32), draw_landmarks

    else:
        # Two-hand mode (126 features)
        left_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)
        right_vec = np.zeros(LANDMARKS_PER_HAND, dtype=np.float32)

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_lm, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                draw_landmarks.append(hand_lm)
                label = handedness.classification[0].label  # 'Left' or 'Right'
                vec = np.array([[p.x, p.y, p.z] for p in hand_lm.landmark], dtype=np.float32).flatten()

                # Note: MediaPipe labels are mirrored (camera mirror effect)
                # 'Left' in MediaPipe = right hand in real life (when image is flipped)
                if label == 'Left':
                    left_vec = vec
                else:
                    right_vec = vec

        # Concatenate: [left_hand(63) | right_hand(63)] = 126 features
        combined = np.concatenate([left_vec, right_vec])
        return combined, draw_landmarks

print(f'✅ MediaPipe hand detector ready ({NUM_HANDS} hand(s) mode)')
print(f'   Features per frame: {NUM_FEATURES}')


✅ MediaPipe hand detector ready (1 hand(s) mode)
   Features per frame: 462


In [ ]:
# ==========================================
# 🎥 OPTIMIZED LIVE WEBCAM PREDICTION CELL
# ==========================================
import cv2
import numpy as np
import mediapipe as mp
import time

# 1. Initialize MediaPipe Hands
mp_hands = mp.solutions.hands
hands_detector = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# 2. Initialize Variables
sequence = []
sentence = []
predictions = []
threshold = 0.85     # Only accept guesses above 85% confidence
frame_counter = 0    # Used to skip frames

# 3. Optimize Webcam Resolution (Reduces lag)
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

print("🎥 Starting live webcam feed... Press 'q' to quit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    frame_counter += 1
    
    # Process the frame with MediaPipe
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    rgb.flags.writeable = False
    results = hands_detector.process(rgb)
    rgb.flags.writeable = True

    # Extract the 126 features (Both Hands)
    left_vec = np.zeros(63, dtype=np.float32)
    right_vec = np.zeros(63, dtype=np.float32)
    
    if results.multi_hand_landmarks and results.multi_handedness:
        for hand_lm, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
            label = handedness.classification[0].label 
            vec = np.array([[p.x, p.y, p.z] for p in hand_lm.landmark]).flatten()
            
            if label == 'Left':
                left_vec = vec
            else:
                right_vec = vec
                
            # Draw skeletons on the camera feed
            mp.solutions.drawing_utils.draw_landmarks(
                frame, hand_lm, mp_hands.HAND_CONNECTIONS)

    # Add the current frame's data to our sequence
    keypoints = np.concatenate([left_vec, right_vec])
    sequence.append(keypoints)
    sequence = sequence[-30:] # Always keep exactly the last 30 frames
    
    # ==========================================
    # 🚀 AI PREDICTION LOGIC
    # ==========================================
    if len(sequence) == 30:
        # OPTIMIZATION: Only run the heavy AI math every 3rd frame
        if frame_counter % 3 == 0:  
            
            # FAST INFERENCE: Call the model directly, do NOT use model.predict()
            res = model(np.expand_dims(sequence, axis=0), training=False)[0].numpy()
            
            predicted_class = np.argmax(res)
            confidence = res[predicted_class]
            predictions.append(predicted_class)
            
            # Stabilization: Ensure the AI predicted the same word 10 times in a row
            if len(predictions) >= 10 and np.unique(predictions[-10:])[0] == predicted_class: 
                if confidence > threshold:
                    # NOTE: Change 'word_labels' to whatever your actual label list is called!
                    word = word_labels[predicted_class] 
                    
                    # Add to sentence if it's a new word
                    if len(sentence) > 0: 
                        if word != sentence[-1]:
                            sentence.append(word)
                    else:
                        sentence.append(word)

        # Keep the sentence short for the screen
        if len(sentence) > 5: 
            sentence = sentence[-5:]

    # Display the predicted sentence at the top of the screen
    cv2.rectangle(frame, (0,0), (640, 40), (245, 117, 16), -1)
    cv2.putText(frame, ' '.join(sentence), (10, 30), 
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)
    
    # Show the video feed
    cv2.imshow('ASL Real-Time Translation', frame)

    # Press 'q' on your keyboard to close the window
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

# Cleanup
cap.release()
cv2.destroyAllWindows()
hands_detector.close()
print("🛑 Webcam feed closed.")


🎥 Starting live webcam feed... Press 'q' to quit.


ValueError: Input 0 of layer "ASL_Word_BiLSTM_v2" is incompatible with the layer: expected shape=(None, 30, 462), found shape=(1, 30, 126)

: 

## Tips

| Issue                      | Solution                                                                                       |
| -------------------------- | ---------------------------------------------------------------------------------------------- |
| **Low FPS**                | Close other apps, reduce `CAMERA_WIDTH`/`CAMERA_HEIGHT`                                        |
| **Wrong predictions**      | Hold the sign steadily for ~2 seconds                                                          |
| **Camera not opening**     | Change `CAMERA_INDEX` to 1 or 2                                                                |
| **Too sensitive**          | Increase `STABILITY_WINDOW` to 4-5                                                             |
| **Not detecting**          | Lower `CONFIDENCE_THRESHOLD` to 0.25                                                           |
| **Too slow between words** | Decrease `COOLDOWN_TIME` to 1.0                                                                |
| **Only 1 hand shown**      | The model auto-detects hand count from its input shape. Retrain with 2 hands for full support. |

### How to perform a sign:

1. Face the camera with your hand(s) clearly visible
2. Perform the sign gesture smoothly
3. Wait for the stability bar to fill up
4. The word will be confirmed and added to the sentence

### Two-Hand Mode Notes:

- If your model was trained with 126 features (2 hands), both hands will be tracked
- Hands are ordered consistently: Left first, Right second
- If only one hand is visible, the other hand's landmarks are zero-padded
- For best results with two-hand signs, keep both hands in the camera frame
